In [0]:
# ============================================================
# SILVER TO GOLD TRANSFORMATION
# Create business-ready aggregations
# ============================================================

# CELL 1: Setup
from pyspark.sql.functions import *
from pyspark.sql.window import Window

storage_account_name = "storagefordeproject1"

silver_base = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"
gold_base = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"

print("✓ Starting Silver to Gold transformation")

In [0]:
# ============================================================
# CELL 2: Read Silver Layer Tables
# ============================================================

print("\n--- Reading Silver Layer ---")

# Dimensions (only current records for dimensions)
dim_customers = spark.read.format("delta").load(f"{silver_base}dim_customers/") \
    .filter(col("is_current") == True)

dim_products = spark.read.format("delta").load(f"{silver_base}dim_products/") \
    .filter(col("is_current") == True)

# Facts
fact_orders = spark.read.format("delta").load(f"{silver_base}fact_orders/")
fact_order_items = spark.read.format("delta").load(f"{silver_base}fact_order_items/")

print(f"✓ Customers: {dim_customers.count()} records")
print(f"✓ Products: {dim_products.count()} records")
print(f"✓ Orders: {fact_orders.count()} records")
print(f"✓ Order Items: {fact_order_items.count()} records")

In [0]:
# ============================================================
# CELL 3: Create Enriched Fact Table
# ============================================================

print("\n--- Creating Enriched Order Facts ---")

# Join orders with customers and order items with products
enriched_orders = fact_orders.alias("o") \
    .join(dim_customers.alias("c"), col("o.customer_id") == col("c.customer_id"), "left") \
    .join(fact_order_items.alias("oi"), col("o.order_id") == col("oi.order_id"), "left") \
    .join(dim_products.alias("p"), col("oi.product_id") == col("p.product_id"), "left") \
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("c.customer_segment"),
        col("c.city").alias("customer_city"),
        col("c.state").alias("customer_state"),
        col("o.order_date"),
        col("o.order_status"),
        col("o.total_amount").alias("order_total"),
        col("oi.product_id"),
        col("p.product_name"),
        col("p.category"),
        col("p.subcategory"),
        col("p.brand"),
        col("oi.quantity"),
        col("oi.unit_price"),
        col("oi.discount_percent"),
        col("oi.line_total"),
        year(col("o.order_date")).alias("order_year"),
        month(col("o.order_date")).alias("order_month"),
        quarter(col("o.order_date")).alias("order_quarter"),
        dayofweek(col("o.order_date")).alias("order_day_of_week")
    )

print(f"✓ Created enriched fact table with {enriched_orders.count()} records")

In [0]:
# ============================================================
# CELL 4: Aggregation 1 - Daily Sales Summary
# ============================================================

print("\n--- Creating Daily Sales Summary ---")

daily_sales = enriched_orders.groupBy(
    "order_date",
    "order_year",
    "order_month",
    "order_quarter"
).agg(
    sum("order_total").alias("total_sales"),
    sum("line_total").alias("total_line_items_value"),
    count("order_id").alias("order_count"),
    countDistinct("customer_id").alias("unique_customers"),
    avg("order_total").alias("avg_order_value"),
    sum("quantity").alias("total_items_sold")
).orderBy("order_date")

# Write to Gold
daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}daily_sales_summary/")

print(f"✓ Daily sales summary created: {daily_sales.count()} days")
display(daily_sales.orderBy(col("order_date").desc()).limit(10))

In [0]:
# ============================================================
# CELL 5: Aggregation 2 - Customer Analytics
# ============================================================

print("\n--- Creating Customer Analytics ---")

customer_analytics = enriched_orders.groupBy(
    "customer_id",
    "customer_segment",
    "customer_city",
    "customer_state"
).agg(
    sum("order_total").alias("lifetime_value"),
    count("order_id").alias("total_orders"),
    sum("quantity").alias("total_items_purchased"),
    avg("order_total").alias("avg_order_value"),
    min("order_date").alias("first_order_date"),
    max("order_date").alias("last_order_date"),
    countDistinct("category").alias("unique_categories_purchased")
)

# Add customer tier based on lifetime value
customer_analytics = customer_analytics.withColumn(
    "customer_tier",
    when(col("lifetime_value") > 5000, "Platinum")
    .when(col("lifetime_value") > 2000, "Gold")
    .when(col("lifetime_value") > 500, "Silver")
    .otherwise("Bronze")
)

# Write to Gold
customer_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}customer_analytics/")

print(f"✓ Customer analytics created: {customer_analytics.count()} customers")
display(customer_analytics.orderBy(col("lifetime_value").desc()).limit(10))

In [0]:
# ============================================================
# CELL 6: Aggregation 3 - Product Performance
# ============================================================

print("\n--- Creating Product Performance ---")

product_performance = enriched_orders.groupBy(
    "product_id",
    "product_name",
    "category",
    "subcategory",
    "brand"
).agg(
    sum("line_total").alias("total_revenue"),
    sum("quantity").alias("total_quantity_sold"),
    count("order_id").alias("order_count"),
    avg("unit_price").alias("avg_selling_price"),
    avg("discount_percent").alias("avg_discount_percent")
).withColumn(
    "revenue_rank",
    dense_rank().over(Window.orderBy(col("total_revenue").desc()))
)

# Write to Gold
product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}product_performance/")

print(f"✓ Product performance created: {product_performance.count()} products")
display(product_performance.orderBy("revenue_rank").limit(10))

In [0]:
# ============================================================
# CELL 7: Aggregation 4 - Category Performance
# ============================================================

print("\n--- Creating Category Performance ---")

category_performance = enriched_orders.groupBy(
    "category",
    "order_year",
    "order_quarter"
).agg(
    sum("line_total").alias("total_revenue"),
    sum("quantity").alias("total_quantity_sold"),
    count("order_id").alias("order_count"),
    countDistinct("customer_id").alias("unique_customers")
).orderBy("order_year", "order_quarter", col("total_revenue").desc())

# Write to Gold
category_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_year", "order_quarter") \
    .save(f"{gold_base}category_performance/")

print(f"✓ Category performance created")
display(category_performance.limit(20))

In [0]:
# ============================================================
# CELL 8: Aggregation 5 - Monthly Trends
# ============================================================

print("\n--- Creating Monthly Trends ---")

monthly_trends = enriched_orders.groupBy(
    "order_year",
    "order_month",
    "customer_segment"
).agg(
    sum("order_total").alias("total_sales"),
    count("order_id").alias("order_count"),
    countDistinct("customer_id").alias("unique_customers"),
    avg("order_total").alias("avg_order_value")
).orderBy("order_year", "order_month")

# Calculate month-over-month growth
window_spec = Window.partitionBy("customer_segment").orderBy("order_year", "order_month")

monthly_trends = monthly_trends.withColumn(
    "prev_month_sales",
    lag("total_sales").over(window_spec)
).withColumn(
    "mom_growth_pct",
    when(col("prev_month_sales").isNotNull(),
         ((col("total_sales") - col("prev_month_sales")) / col("prev_month_sales") * 100)
    ).otherwise(None)
)

# Write to Gold
monthly_trends.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}monthly_trends/")

print(f"✓ Monthly trends created")
display(monthly_trends.limit(20))

In [0]:
# ============================================================
# CELL 9: Summary
# ============================================================

print("\n" + "="*60)
print("GOLD LAYER SUMMARY")
print("="*60)

gold_tables = [
    "daily_sales_summary",
    "customer_analytics",
    "product_performance",
    "category_performance",
    "monthly_trends"
]

for table in gold_tables:
    path = f"{gold_base}{table}/"
    count = spark.read.format("delta").load(path).count()
    print(f"✓ {table}: {count} records")

print("\n✓ SILVER TO GOLD TRANSFORMATION COMPLETE!")
print("="*60)

In [0]:
# ============================================================
# CELL 10: Sample Queries for Validation
# ============================================================

print("\n--- SAMPLE ANALYTICS QUERIES ---\n")

# Query 1: Top 10 customers by lifetime value
print("Top 10 Customers by Lifetime Value:")
top_customers = spark.read.format("delta").load(f"{gold_base}customer_analytics/") \
    .orderBy(col("lifetime_value").desc()) \
    .limit(10)
display(top_customers)

# Query 2: Top 10 products by revenue
print("\nTop 10 Products by Revenue:")
top_products = spark.read.format("delta").load(f"{gold_base}product_performance/") \
    .orderBy(col("total_revenue").desc()) \
    .limit(10)
display(top_products)

# Query 3: Sales trend last 30 days
print("\nSales Trend - Last 30 Days:")
recent_sales = spark.read.format("delta").load(f"{gold_base}daily_sales_summary/") \
    .orderBy(col("order_date").desc()) \
    .limit(30)
display(recent_sales)

print("\n✓ All validation queries complete!")